# Explore World Bank Data


## Pre-requisites

1. Run the `prepare_world_bank_data.ipynb` to download data via the [Data 360 API](https://data360.worldbank.org/en/api) and prepare the data in a DuckDB database.

In [1]:
from pathlib import Path

from data_wranglers import WorldBankDataWrangler
from charting_helpers import WorldBankChartingHelper

In [2]:
DUCKDB_PATH = Path("../../databases") / "world_bank.db"
wrangler = WorldBankDataWrangler(DUCKDB_PATH)

In [3]:
countries = wrangler.get_countries()
countries

country_code,iso2_code,country_name,region,capital_city,longitude,latitude
str,str,str,str,str,f64,f64
"""AFG""","""AF""","""Afghanistan""","""Middle East, North Africa, Afg…","""Kabul""",69.1761,34.5228
"""ALB""","""AL""","""Albania""","""Europe & Central Asia""","""Tirane""",19.8172,41.3317
"""DZA""","""DZ""","""Algeria""","""Middle East, North Africa, Afg…","""Algiers""",3.05097,36.7397
"""ASM""","""AS""","""American Samoa""","""East Asia & Pacific""","""Pago Pago""",-170.691,-14.2846
"""AND""","""AD""","""Andorra""","""Europe & Central Asia""","""Andorra la Vella""",1.5218,42.5075
…,…,…,…,…,…,…
"""VIR""","""VI""","""Virgin Islands (U.S.)""","""Latin America & Caribbean ""","""Charlotte Amalie""",-64.8963,18.3358
"""PSE""","""PS""","""West Bank and Gaza""","""Middle East, North Africa, Afg…","""""",null,null
"""YEM""","""YE""","""Yemen, Rep.""","""Middle East, North Africa, Afg…","""Sana'a""",44.2075,15.352


## World Wealth and Health

Query and transform data combining GDP per capita, life expectancy, and population using DuckDB + Polars, then visualise changes over time.

In [4]:
world_wealth_and_health = wrangler.get_wealth_and_health_data()
world_wealth_and_health.head(3)

country_code,country_name,region,year,longitude,latitude,gdp_usd_per_capita,life_expectancy,population,population_in_millions
str,str,str,i64,f64,f64,f64,f64,f64,f64
"""ALB""","""Albania""","""Europe & Central Asia""",1980,19.8172,41.3317,590.607738,69.903,2.671997e6,2.67
"""ALB""","""Albania""","""Europe & Central Asia""",1981,19.8172,41.3317,663.294208,70.145,2.726056e6,2.73
"""ALB""","""Albania""","""Europe & Central Asia""",1982,19.8172,41.3317,668.454504,70.425,2.784278e6,2.78


In [5]:
WorldBankChartingHelper.chart_country_locations(world_wealth_and_health)

In [6]:
WorldBankChartingHelper.chart_wealth_and_health(world_wealth_and_health)

## Population Analysis

In [7]:
COUNTRY = "United Kingdom"
uk_population = wrangler.get_population_analysis(COUNTRY)
uk_population.head(3)

year,population_in_millions,population_change_percentage,color
i64,f64,f64,str
1970,55.66,null,"""red"""
1971,55.9,0.431189,"""green"""
1972,56.09,0.339893,"""green"""


In [8]:
WorldBankChartingHelper.chart_population_analysis(uk_population, COUNTRY)

## Compare a Metric Across Countries

In [9]:
import polars as pl

filtered = (
    world_wealth_and_health
    .filter(
        pl.col("country_name").is_in(["United Kingdom", "France", "Germany", "United States", "Japan", "Brazil"])
        & pl.col("year").is_between(1970, 2024)
    )
    .sort(["country_name", "year"])
)

WorldBankChartingHelper.chart_metric_by_country(filtered, "life_expectancy")